# 🛠️ Notebook 2: Online Stock Brokerage — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/online-stock-brokerage
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from __future__ import annotations
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from itertools import count

@dataclass
class Stock:
    symbol: str
    price: float   # "current" market price — kept simple

class Side(Enum):
    BUY = "buy"; SELL = "sell"

class OrderStatus(Enum):
    PENDING = "pending"; FILLED = "filled"; CANCELLED = "cancelled"

@dataclass
class Position:
    symbol: str
    qty: int = 0
    avg_price: float = 0.0

@dataclass
class Portfolio:
    positions: dict[str, Position] = field(default_factory=dict)
    def apply(self, symbol: str, qty_delta: int, price: float):
        pos = self.positions.setdefault(symbol, Position(symbol))
        if qty_delta > 0:
            new_qty = pos.qty + qty_delta
            pos.avg_price = (pos.avg_price * pos.qty + price * qty_delta) / new_qty
            pos.qty = new_qty
        else:
            pos.qty += qty_delta   # selling doesn't change avg
        if pos.qty == 0: self.positions.pop(symbol, None)

@dataclass
class Account:
    id: int
    cash: float
    portfolio: Portfolio = field(default_factory=Portfolio)


In [ ]:
_oids = count(1)

class Order(ABC):
    def __init__(self, account: Account, stock: Stock, side: Side, qty: int):
        self.id = next(_oids)
        self.account = account
        self.stock = stock
        self.side = side
        self.qty = qty
        self.status = OrderStatus.PENDING
    @abstractmethod
    def can_fill(self, market_price: float) -> bool: ...
    @abstractmethod
    def fill_price(self, market_price: float) -> float: ...

class MarketOrder(Order):
    def can_fill(self, market_price): return True
    def fill_price(self, market_price): return market_price

class LimitOrder(Order):
    def __init__(self, account, stock, side, qty, limit_price):
        super().__init__(account, stock, side, qty); self.limit_price = limit_price
    def can_fill(self, market_price):
        return (market_price <= self.limit_price) if self.side == Side.BUY \
               else (market_price >= self.limit_price)
    def fill_price(self, market_price): return market_price

class StopOrder(Order):
    def __init__(self, account, stock, side, qty, stop_price):
        super().__init__(account, stock, side, qty); self.stop_price = stop_price
    def can_fill(self, market_price):
        return (market_price >= self.stop_price) if self.side == Side.BUY \
               else (market_price <= self.stop_price)
    def fill_price(self, market_price): return market_price


In [ ]:
@dataclass
class Trade:
    order_id: int
    symbol: str
    side: Side
    qty: int
    price: float
    ts: datetime = field(default_factory=datetime.utcnow)

class Exchange:
    def __init__(self):
        self.book: list[Order] = []
        self.trades: list[Trade] = []

    def place(self, order: Order):
        self.book.append(order)
        self._try_fill(order)

    def tick(self, symbol: str, new_price: float):
        # market moves; try to fill resting orders
        for o in list(self.book):
            if o.stock.symbol == symbol and o.status == OrderStatus.PENDING:
                o.stock.price = new_price
                self._try_fill(o)

    def _try_fill(self, o: Order):
        mp = o.stock.price
        if not o.can_fill(mp): return
        fp = o.fill_price(mp); cost = fp * o.qty
        if o.side == Side.BUY:
            if o.account.cash < cost: return              # not enough cash, still pending
            o.account.cash -= cost
            o.account.portfolio.apply(o.stock.symbol, +o.qty, fp)
        else:
            pos = o.account.portfolio.positions.get(o.stock.symbol)
            if not pos or pos.qty < o.qty: return          # can't short in this toy
            o.account.cash += cost
            o.account.portfolio.apply(o.stock.symbol, -o.qty, fp)
        o.status = OrderStatus.FILLED
        self.trades.append(Trade(o.id, o.stock.symbol, o.side, o.qty, fp))
        self.book.remove(o)

ex = Exchange()
aapl = Stock("AAPL", 180)
alice = Account(1, cash=10_000)

ex.place(MarketOrder(alice, aapl, Side.BUY, 10))         # fills now at 180
ex.place(LimitOrder(alice, aapl, Side.BUY, 5, limit_price=170))  # waits
ex.tick("AAPL", 168)                                     # price drops, limit triggers
print("Cash:", alice.cash, "Positions:", alice.portfolio.positions)
print("Trades:", ex.trades)


### Try it
- Add an `OrderBook` (bids/asks) with proper price-time priority matching.
- Add `cancel(order_id)` transitioning to `CANCELLED`.
- Add a `Quote` observer printing mark-to-market P&L on each tick.